# Policy Performance Sandbox

This notebook is a local shadow evaluator for the **embedded GRE policies**. It is meant to estimate possible performance, not official leaderboard performance.

## What This Does
- Uses the embedded `StudentPricingPolicy` and `StudentMatchingPolicy` directly in this notebook.
- Replays the historical arrival stream in `data/training_data.csv`.
- Uses an empirical conversion proxy to estimate whether riders would accept your quoted price.
- Simulates matching and dispatch under a configurable batching horizon.
- Reports directional metrics such as quote level, conversion rate, shared-match rate, wait time, revenue, cost, and profit proxy.

In [1]:
import json
import pickle
import copy
import math
import time
from collections import defaultdict

import numpy as np
import pandas as pd

from rider import rider
from utils import populate_shared_ride_lengths

DATA_PATH = "data/training_data.csv"
COST_PER_MILE = 0.70
MAX_WAIT_SECONDS = 120
N_REPLICATIONS = 1
RANDOM_SEED = 42
SAMPLE_FRAC = 1.0
WAIT_GRID = (15, 60, 120)


In [2]:
# # Student Policies
# Now that you know how model cost, conversion, demand, and optimize price and matching, we can move on the final part of this game, **building your policies!**. Below we have set up skeleton code for you to build your policies. Make when you finish to run the last code cell to upload your policy.

# In[2]:


# Load in all your libraries and packages here!
# NOTE: you are allowed to use packages loaded below ONLY. If you want to use any other packages, please contact the TA.
import pandas as pd
import numpy as np
import itertools
import math
import random
import copy
import time
import pandas as pd
import haversine
import unittest
import pickle
import collections

from utils import * # this imports all helper functions in utils.py

# TODO: Replace the string "TeamName" below with your own team's name in camel case. For e.g. "AwesomeTeam"
# DO NOT RENAME THE FILE, the file name should be "{TEAM_NAME}_Policies.py"
TEAM_NAME = "GRE"


# # Pricing Policy

# Feel free to add more functions to `StudentPricingPolicy` but **DO NOT** modify the initalization and static methods. Your code will only be graded based on the output of your pricing function.

# In[3]:


class StudentPricingPolicy:
    """Pricing policy that uses current queue geometry more than historical route frequency."""

    BASE_PRICE = 0.725
    PRICE_FLOOR = 0.62
    PRICE_CEILING = 0.81
    PAIR_CACHE = {}

    # DO NOT MODIFY
    def __init__(self, c = 0.70):
        self.c = c

    # DO NOT MODIFY
    @staticmethod
    def get_name():
        return TEAM_NAME

    @staticmethod
    def _clip(value, lower=0.0, upper=1.0):
        return max(lower, min(upper, value))

    @staticmethod
    def _safe_ratio(numerator, denominator):
        if denominator <= 0:
            return 0.0
        return float(numerator) / float(denominator)

    @staticmethod
    def _rider_signature(rider):
        return (
            round(float(rider.pickup_lat), 6),
            round(float(rider.pickup_lon), 6),
            round(float(rider.dropoff_lat), 6),
            round(float(rider.dropoff_lon), 6),
            int(rider.pickup_area),
            int(rider.dropoff_area),
            round(float(rider.arrival_time), 3),
        )

    @classmethod
    def _pair_metrics(cls, rider_i, rider_j):
        sig_i = cls._rider_signature(rider_i)
        sig_j = cls._rider_signature(rider_j)
        cache_key = (sig_i, sig_j) if sig_i <= sig_j else (sig_j, sig_i)
        cached = cls.PAIR_CACHE.get(cache_key)
        if cached is not None:
            return cached

        origin_i = (rider_i.pickup_lat, rider_i.pickup_lon)
        dest_i = (rider_i.dropoff_lat, rider_i.dropoff_lon)
        origin_j = (rider_j.pickup_lat, rider_j.pickup_lon)
        dest_j = (rider_j.dropoff_lat, rider_j.dropoff_lon)

        trip_length, shared_length, i_solo_length, j_solo_length, _ = populate_shared_ride_lengths(
            origin_i,
            dest_i,
            origin_j,
            dest_j,
        )

        total_length = rider_i.solo_length + rider_j.solo_length
        min_length = min(rider_i.solo_length, rider_j.solo_length)
        solo_ratio_i = cls._safe_ratio(i_solo_length, rider_i.solo_length)
        solo_ratio_j = cls._safe_ratio(j_solo_length, rider_j.solo_length)

        metrics = {
            "trip_ratio": cls._safe_ratio(trip_length, total_length),
            "shared_ratio": cls._safe_ratio(shared_length, min_length),
            "solo_ratio_max": max(solo_ratio_i, solo_ratio_j),
            "absolute_savings": max(0.0, total_length - trip_length),
            "savings_ratio": max(0.0, 1.0 - cls._safe_ratio(trip_length, total_length)),
            "same_pickup": int(rider_i.pickup_area == rider_j.pickup_area),
            "same_dropoff": int(rider_i.dropoff_area == rider_j.dropoff_area),
        }

        if len(cls.PAIR_CACHE) > 25000:
            cls.PAIR_CACHE.clear()
        cls.PAIR_CACHE[cache_key] = metrics
        return metrics

    @classmethod
    def _shareability_snapshot(cls, state, rider):
        same_route = 0
        same_pickup = 0
        same_dropoff = 0
        viable_count = 0
        strong_count = 0
        best_value = 0.0

        for waiting_rider in state:
            if int(waiting_rider.pickup_area) == int(rider.pickup_area):
                same_pickup += 1
            if int(waiting_rider.dropoff_area) == int(rider.dropoff_area):
                same_dropoff += 1
            if int(waiting_rider.pickup_area) == int(rider.pickup_area) and int(waiting_rider.dropoff_area) == int(rider.dropoff_area):
                same_route += 1

            metrics = cls._pair_metrics(rider, waiting_rider)
            if metrics["shared_ratio"] <= 0.0:
                continue
            if metrics["savings_ratio"] < 0.10 and metrics["absolute_savings"] < 0.25:
                continue
            if metrics["solo_ratio_max"] > 0.80 and metrics["absolute_savings"] < 0.45:
                continue

            value = (
                1.80 * metrics["savings_ratio"]
                + 0.80 * min(metrics["shared_ratio"], 1.0)
                + 0.45 * max(0.0, 1.0 - metrics["solo_ratio_max"])
                + 0.12 * metrics["absolute_savings"]
                + 0.05 * metrics["same_pickup"]
                + 0.05 * metrics["same_dropoff"]
            )
            viable_count += 1
            if value >= 0.95:
                strong_count += 1
            best_value = max(best_value, value)

        return {
            "same_route": same_route,
            "same_pickup": same_pickup,
            "same_dropoff": same_dropoff,
            "viable_count": viable_count,
            "strong_count": strong_count,
            "best_value": best_value,
        }

    @classmethod
    def _baseline_price(cls, rider):
        price = cls.BASE_PRICE

        if rider.solo_length < 1.5:
            price -= 0.040
        elif rider.solo_length < 3.0:
            price -= 0.018
        elif rider.solo_length > 8.5:
            price += 0.032
        elif rider.solo_length > 6.0:
            price += 0.018

        return price

    def pricing_function(self, state, rider):
        """
        Returns the price of the given rider in the given state

        Parameters
        ----------
        state: list
            list of rider(s) (object) waiting in the state
        rider: object
            An incoming rider

        Returns
        -------
        float
            The price of the rider: must be in [0, 1]
        """
        price = self._baseline_price(rider)
        state_size = len(state)

        if state_size == 0:
            price += 0.020
            return float(self._clip(price, self.PRICE_FLOOR, self.PRICE_CEILING))

        snapshot = self._shareability_snapshot(state, rider)

        if snapshot["best_value"] >= 1.10:
            price -= 0.028
        elif snapshot["best_value"] >= 0.88:
            price -= 0.018
        elif snapshot["best_value"] >= 0.68:
            price -= 0.010

        if snapshot["same_route"] > 0:
            price -= 0.010
        elif snapshot["same_pickup"] > 0 and snapshot["same_dropoff"] > 0:
            price -= 0.008

        if snapshot["viable_count"] >= 2:
            price -= 0.008
        if snapshot["strong_count"] >= 2:
            price -= 0.006

        if state_size >= 12 and snapshot["viable_count"] == 0:
            price += 0.008
        if state_size >= 30 and snapshot["viable_count"] == 0:
            price += 0.008
        elif state_size >= 30 and snapshot["viable_count"] >= 3:
            price -= 0.006

        return float(self._clip(price, self.PRICE_FLOOR, self.PRICE_CEILING))


# # Matching Policy

# Again, feel free to add more functions to `StudentMatchingPolicy` but **DO NOT** modify the initalization and static methods. Your code will only be graded based on the output of your matching function.

# In[4]:


class StudentMatchingPolicy:
    # DO NOT MODIFY
    def __init__(self, c = 0.70):
        self.c = c

    # DO NOT MODIFY
    @staticmethod
    def get_name():
        return TEAM_NAME

    @staticmethod
    def clamp(x, lo, hi):
        return max(lo, min(hi, x))

    @staticmethod
    def get_origin(r):
        return (r.pickup_lat, r.pickup_lon)

    @staticmethod
    def get_destination(r):
        return (r.dropoff_lat, r.dropoff_lon)

    @staticmethod
    def quick_pair_score(r1, r2):
        """
        Cheap compatibility proxy used before exact routing.
        Lower is better.
        """
        pickup_gap = abs(r1.pickup_area - r2.pickup_area)
        dropoff_gap = abs(r1.dropoff_area - r2.dropoff_area)
        time_gap = abs(r1.arrival_time - r2.arrival_time)
        length_gap = abs(r1.solo_length - r2.solo_length)

        return 2.0 * pickup_gap + 2.0 * dropoff_gap + 0.003 * time_gap + 0.2 * length_gap

    def candidate_exact_match_score(self, rider, waiting_rider):
        """
        Exact pair evaluation using the provided helper.
        """
        trip_length, shared_length, i_solo_length, j_solo_length, trip_order = populate_shared_ride_lengths(
            self.get_origin(rider), self.get_destination(rider),
            self.get_origin(waiting_rider), self.get_destination(waiting_rider)
        )

        solo_sum = rider.solo_length + waiting_rider.solo_length
        saving = solo_sum - trip_length
        saving_ratio = saving / max(solo_sum, 1e-6)
        shared_ratio = shared_length / max(trip_length, 1e-6)
        detour_penalty = (i_solo_length + j_solo_length) / max(trip_length, 1e-6)

        score = 2.2 * saving_ratio + 0.35 * shared_ratio + 0.18 * saving - 0.18 * detour_penalty

        return {
            "saving": saving,
            "saving_ratio": saving_ratio,
            "shared_ratio": shared_ratio,
            "detour_penalty": detour_penalty,
            "score": score,
        }

    def matching_function(self, state, rider):
        if len(state) == 0:
            return None

        max_candidates = 15
        base_min_saving_ratio = 0.12
        base_min_absolute_saving = 0.35
        large_queue_threshold = 20

        # Stage 1: cheap shortlist
        candidates = sorted(state, key=lambda w: self.quick_pair_score(rider, w))[:max_candidates]

        min_saving_ratio = base_min_saving_ratio
        min_absolute_saving = base_min_absolute_saving

        # Large queue => match more aggressively
        if len(state) >= large_queue_threshold:
            min_saving_ratio -= 0.02
            min_absolute_saving -= 0.08

        # Late arrival => waiting is less attractive
        if rider.arrival_time > 3300:
            min_saving_ratio -= 0.015
            min_absolute_saving -= 0.05

        min_saving_ratio = max(0.05, min_saving_ratio)
        min_absolute_saving = max(0.12, min_absolute_saving)

        best_candidate = None
        best_score = -1e18

        # Stage 2: exact evaluation
        for w in candidates:
            stats = self.candidate_exact_match_score(rider, w)

            saving = stats["saving"]
            saving_ratio = stats["saving_ratio"]
            score = stats["score"]

            if saving_ratio >= min_saving_ratio and saving >= min_absolute_saving:
                if score > best_score:
                    best_score = score
                    best_candidate = w

        return best_candidate

print("Loaded pricing policy:", StudentPricingPolicy.get_name())
print("Loaded matching policy:", StudentMatchingPolicy.get_name())


Loaded pricing policy: GRE
Loaded matching policy: GRE


In [3]:
df = pd.read_csv(DATA_PATH).sort_values(["arrival_week", "arrival_time", "rider_id"]).reset_index(drop=True)

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_SEED).sort_values(["arrival_week", "arrival_time", "rider_id"]).reset_index(drop=True)

PRICE_BINS = np.linspace(0.0, 1.0, 21)
LENGTH_BINS = [0.0, 1.5, 2.5, 4.0, 6.0, 9.0, float("inf")]
GLOBAL_CONVERSION = float(df["convert_or_not"].mean())

def price_bucket(price):
    clipped = float(np.clip(price, 0.0, 1.0))
    idx = int(np.digitize([clipped], PRICE_BINS[1:-1], right=False)[0])
    return idx

def length_bucket(length):
    idx = int(np.digitize([float(length)], LENGTH_BINS[1:-1], right=False)[0])
    return idx

def smoothed_rate(table, key, base_rate, strength):
    if key not in table:
        return base_rate
    count, mean = table[key]
    return (count * mean + strength * base_rate) / (count + strength)

def build_rate_table(group_cols, dataframe):
    grouped = dataframe.groupby(group_cols)["convert_or_not"].agg(["count", "mean"]).reset_index()
    table = {}
    for row in grouped.itertuples(index=False):
        key = tuple(getattr(row, col) for col in group_cols)
        table[key] = (int(row.count), float(row.mean))
    return table

df_proxy = df.copy()
df_proxy["price_bucket"] = df_proxy["quoted_price"].apply(price_bucket)
df_proxy["length_bucket"] = df_proxy["solo_length"].apply(length_bucket)
accepted_df = df_proxy[df_proxy["convert_or_not"] == 1].copy()

price_bucket_summary = df_proxy.groupby("price_bucket")["convert_or_not"].agg(["count", "mean"]).reset_index()
PRICE_BUCKET_CENTERS = np.array(
    [(PRICE_BINS[int(idx)] + PRICE_BINS[int(idx) + 1]) / 2.0 for idx in price_bucket_summary["price_bucket"]],
    dtype=float,
)
PRICE_BUCKET_RATES = price_bucket_summary["mean"].to_numpy(dtype=float).copy()
for idx in range(1, len(PRICE_BUCKET_RATES)):
    PRICE_BUCKET_RATES[idx] = min(PRICE_BUCKET_RATES[idx], PRICE_BUCKET_RATES[idx - 1])
GLOBAL_BUCKET_BASE_RATES = {
    int(bucket): max(0.01, float(rate))
    for bucket, rate in zip(price_bucket_summary["price_bucket"], PRICE_BUCKET_RATES)
}

LENGTH_PRICE_TABLE = build_rate_table(["length_bucket", "price_bucket"], df_proxy)
ROUTE_PRICE_TABLE = build_rate_table(["pickup_area", "dropoff_area", "price_bucket"], df_proxy)
ROUTE_LENGTH_PRICE_TABLE = build_rate_table(["pickup_area", "dropoff_area", "length_bucket", "price_bucket"], df_proxy)

ACCEPTED_ROUTE_PRICE_CAP = accepted_df.groupby(["pickup_area", "dropoff_area"])["quoted_price"].max().to_dict()
ACCEPTED_LENGTH_PRICE_CAP = accepted_df.groupby("length_bucket")["quoted_price"].max().to_dict()
GLOBAL_MAX_ACCEPTED_QUOTE = float(accepted_df["quoted_price"].max())


def continuous_price_probability(quoted_price):
    return float(
        np.interp(
            [float(quoted_price)],
            PRICE_BUCKET_CENTERS,
            PRICE_BUCKET_RATES,
            left=PRICE_BUCKET_RATES[0],
            right=PRICE_BUCKET_RATES[-1],
        )[0]
    )


def estimate_conversion_probability(row_like, quoted_price):
    p_bucket = price_bucket(quoted_price)
    l_bucket = length_bucket(row_like["solo_length"])
    route_key = (int(row_like["pickup_area"]), int(row_like["dropoff_area"]))

    prob = GLOBAL_BUCKET_BASE_RATES[p_bucket]
    prob = smoothed_rate(LENGTH_PRICE_TABLE, (l_bucket, p_bucket), prob, 18.0)
    prob = smoothed_rate(ROUTE_PRICE_TABLE, route_key + (p_bucket,), prob, 12.0)
    prob = smoothed_rate(ROUTE_LENGTH_PRICE_TABLE, route_key + (l_bucket, p_bucket), prob, 6.0)

    # Preserve within-bucket price sensitivity instead of treating the whole bucket as flat.
    prob *= continuous_price_probability(quoted_price) / max(GLOBAL_BUCKET_BASE_RATES[p_bucket], 1e-6)

    # Penalize quotes that exceed historically supported accepted prices for this route/length.
    route_cap = ACCEPTED_ROUTE_PRICE_CAP.get(route_key, GLOBAL_MAX_ACCEPTED_QUOTE)
    length_cap = ACCEPTED_LENGTH_PRICE_CAP.get(l_bucket, GLOBAL_MAX_ACCEPTED_QUOTE)
    support_cap = min(max(route_cap, length_cap), GLOBAL_MAX_ACCEPTED_QUOTE)

    if quoted_price > support_cap:
        prob *= math.exp(-18.0 * (float(quoted_price) - support_cap))
    if quoted_price > GLOBAL_MAX_ACCEPTED_QUOTE + 0.01:
        prob *= 0.25

    return float(np.clip(prob, 0.01, 0.98))


def row_to_rider(row_like):
    return rider(
        int(row_like["arrival_week"]),
        float(row_like["arrival_time"]),
        float(row_like["pickup_lat"]),
        float(row_like["pickup_lon"]),
        float(row_like["dropoff_lat"]),
        float(row_like["dropoff_lon"]),
        int(row_like["pickup_area"]),
        int(row_like["dropoff_area"]),
    )


def pair_cost_lengths(rider_i, rider_j):
    origin_i = (rider_i.pickup_lat, rider_i.pickup_lon)
    dest_i = (rider_i.dropoff_lat, rider_i.dropoff_lon)
    origin_j = (rider_j.pickup_lat, rider_j.pickup_lon)
    dest_j = (rider_j.dropoff_lat, rider_j.dropoff_lon)

    trip_length, shared_length, i_solo_length, j_solo_length, _ = populate_shared_ride_lengths(
        origin_i,
        dest_i,
        origin_j,
        dest_j,
    )

    return float(i_solo_length + shared_length / 2.0), float(j_solo_length + shared_length / 2.0), float(trip_length)

print(f"Loaded {len(df):,} riders for shadow evaluation.")
print(f"Global historical conversion rate: {GLOBAL_CONVERSION:.3f}")
print(f"Global max accepted quote: {GLOBAL_MAX_ACCEPTED_QUOTE:.3f}")

Loaded 11,788 riders for shadow evaluation.
Global historical conversion rate: 0.582
Global max accepted quote: 0.845


In [4]:
def initialize_metrics():
    return defaultdict(float)


def dispatch_solo(record, dispatch_time, metrics):
    wait_time = max(0.0, float(dispatch_time) - float(record["arrival_time"]))
    revenue = float(record["price"]) * float(record["solo_length"])
    cost = COST_PER_MILE * float(record["solo_length"])

    metrics["converted_riders"] += 1
    metrics["solo_dispatches"] += 1
    metrics["revenue"] += revenue
    metrics["cost"] += cost
    metrics["profit"] += revenue - cost
    metrics["total_wait_time"] += wait_time


def dispatch_shared(incoming_record, waiting_record, current_time, metrics):
    cost_incoming, cost_waiting, trip_length = pair_cost_lengths(incoming_record["rider"], waiting_record["rider"])

    revenue_incoming = float(incoming_record["price"]) * float(incoming_record["solo_length"])
    revenue_waiting = float(waiting_record["price"]) * float(waiting_record["solo_length"])

    wait_incoming = max(0.0, float(current_time) - float(incoming_record["arrival_time"]))
    wait_waiting = max(0.0, float(current_time) - float(waiting_record["arrival_time"]))

    metrics["converted_riders"] += 2
    metrics["shared_riders"] += 2
    metrics["matched_pairs"] += 1
    metrics["revenue"] += revenue_incoming + revenue_waiting
    metrics["cost"] += COST_PER_MILE * (cost_incoming + cost_waiting)
    metrics["profit"] += revenue_incoming + revenue_waiting - COST_PER_MILE * (cost_incoming + cost_waiting)
    metrics["total_wait_time"] += wait_incoming + wait_waiting
    metrics["total_trip_length"] += trip_length


def flush_expired(queue, current_time, max_wait_seconds, metrics):
    still_waiting = []
    for record in queue:
        if float(current_time) - float(record["arrival_time"]) >= max_wait_seconds:
            dispatch_solo(record, record["arrival_time"] + max_wait_seconds, metrics)
        else:
            still_waiting.append(record)
    return still_waiting


def finalize_queue(queue, max_wait_seconds, metrics):
    for record in queue:
        dispatch_solo(record, record["arrival_time"] + max_wait_seconds, metrics)


def summarize_metrics(metrics):
    summary = dict(metrics)
    requests = max(summary.get("requests", 0.0), 1.0)
    converted = max(summary.get("converted_riders", 0.0), 1.0)

    summary["avg_quote"] = summary.get("quoted_price_sum", 0.0) / requests
    summary["expected_conversion_rate"] = summary.get("convert_prob_sum", 0.0) / requests
    summary["simulated_conversion_rate"] = summary.get("converted_riders", 0.0) / requests
    summary["shared_rate_given_conversion"] = summary.get("shared_riders", 0.0) / converted
    summary["avg_wait_time"] = summary.get("total_wait_time", 0.0) / converted
    summary["profit_per_request"] = summary.get("profit", 0.0) / requests
    summary["profit_per_converted_rider"] = summary.get("profit", 0.0) / converted
    summary["revenue_per_request"] = summary.get("revenue", 0.0) / requests
    summary["cost_per_request"] = summary.get("cost", 0.0) / requests

    return summary


In [5]:
def simulate_once(dataframe, max_wait_seconds=MAX_WAIT_SECONDS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    pricing_policy = StudentPricingPolicy()
    matching_policy = StudentMatchingPolicy()
    metrics = initialize_metrics()

    queue = []
    current_week = None

    for row in dataframe.itertuples(index=False):
        row_dict = row._asdict()

        if current_week is None:
            current_week = int(row_dict["arrival_week"])
        elif int(row_dict["arrival_week"]) != current_week:
            finalize_queue(queue, max_wait_seconds, metrics)
            queue = []
            current_week = int(row_dict["arrival_week"])

        current_time = float(row_dict["arrival_time"])
        queue = flush_expired(queue, current_time, max_wait_seconds, metrics)

        incoming_rider = row_to_rider(row_dict)
        state = [record["rider"] for record in queue]
        quoted_price = float(pricing_policy.pricing_function(state, incoming_rider))
        convert_prob = estimate_conversion_probability(row_dict, quoted_price)

        metrics["requests"] += 1
        metrics["quoted_price_sum"] += quoted_price
        metrics["convert_prob_sum"] += convert_prob

        if rng.random() >= convert_prob:
            continue

        incoming_record = {
            "rider": incoming_rider,
            "arrival_week": int(row_dict["arrival_week"]),
            "arrival_time": current_time,
            "solo_length": float(row_dict["solo_length"]),
            "price": quoted_price,
            "rider_id": int(row_dict["rider_id"]),
        }

        matched_waiting_rider = matching_policy.matching_function(state, incoming_rider)
        matched_index = None

        if matched_waiting_rider is not None:
            for idx, waiting_record in enumerate(queue):
                if waiting_record["rider"] is matched_waiting_rider:
                    matched_index = idx
                    break

        if matched_index is not None:
            waiting_record = queue.pop(matched_index)
            dispatch_shared(incoming_record, waiting_record, current_time, metrics)
        else:
            queue.append(incoming_record)

    finalize_queue(queue, max_wait_seconds, metrics)
    return summarize_metrics(metrics)


def evaluate_policy(wait_grid=WAIT_GRID, n_replications=N_REPLICATIONS, base_seed=RANDOM_SEED):
    rows = []
    for max_wait_seconds in wait_grid:
        replications = []
        for rep in range(n_replications):
            replications.append(simulate_once(df, max_wait_seconds=max_wait_seconds, seed=base_seed + rep))

        rep_df = pd.DataFrame(replications)
        summary = rep_df.mean(numeric_only=True).to_dict()
        summary["max_wait_seconds"] = max_wait_seconds
        summary["n_replications"] = n_replications
        rows.append(summary)

    columns = [
        "max_wait_seconds",
        "n_replications",
        "avg_quote",
        "expected_conversion_rate",
        "simulated_conversion_rate",
        "shared_rate_given_conversion",
        "avg_wait_time",
        "revenue",
        "cost",
        "profit",
        "profit_per_request",
        "profit_per_converted_rider",
        "matched_pairs",
    ]

    return pd.DataFrame(rows)[columns]


In [6]:
def historical_cost_length(row_like, rider_lookup):
    if pd.isna(row_like["matching_outcome"]):
        return float(row_like["solo_length"])

    matched_rider_id = int(row_like["matching_outcome"])
    if matched_rider_id not in rider_lookup:
        return float(row_like["solo_length"])

    matched_row = rider_lookup[matched_rider_id]

    origin_i = (float(row_like["pickup_lat"]), float(row_like["pickup_lon"]))
    dest_i = (float(row_like["dropoff_lat"]), float(row_like["dropoff_lon"]))
    origin_j = (float(matched_row["pickup_lat"]), float(matched_row["pickup_lon"]))
    dest_j = (float(matched_row["dropoff_lat"]), float(matched_row["dropoff_lon"]))

    trip_length, shared_length, i_solo_length, j_solo_length, _ = populate_shared_ride_lengths(origin_i, dest_i, origin_j, dest_j)
    return float(i_solo_length + shared_length / 2.0)


def historical_baseline_summary(dataframe):
    rider_lookup = dataframe.set_index("rider_id")[["pickup_lat", "pickup_lon", "dropoff_lat", "dropoff_lon"]].to_dict("index")

    converted = dataframe[dataframe["convert_or_not"] == 1].copy()
    converted["revenue"] = converted["quoted_price"] * converted["solo_length"]
    converted["cost_length"] = converted.apply(lambda row: historical_cost_length(row, rider_lookup), axis=1)
    converted["cost"] = COST_PER_MILE * converted["cost_length"]
    converted["profit"] = converted["revenue"] - converted["cost"]

    summary = {
        "requests": float(len(dataframe)),
        "avg_quote": float(dataframe["quoted_price"].mean()),
        "expected_conversion_rate": float(dataframe["convert_or_not"].mean()),
        "simulated_conversion_rate": float(dataframe["convert_or_not"].mean()),
        "shared_rate_given_conversion": float(converted["matching_outcome"].notna().mean()) if len(converted) else 0.0,
        "avg_wait_time": float(converted["waiting_time"].fillna(0.0).mean()) if len(converted) else 0.0,
        "revenue": float(converted["revenue"].sum()),
        "cost": float(converted["cost"].sum()),
        "profit": float(converted["profit"].sum()),
        "profit_per_request": float(converted["profit"].sum() / max(len(dataframe), 1)),
        "profit_per_converted_rider": float(converted["profit"].sum() / max(len(converted), 1)),
        "matched_pairs": float(converted["matching_outcome"].notna().sum() / 2.0),
    }

    return pd.Series(summary, name="historical_baseline")

baseline = historical_baseline_summary(df)
baseline


requests                        11788.000000
avg_quote                           0.558944
expected_conversion_rate            0.581948
simulated_conversion_rate           0.581948
shared_rate_given_conversion        0.798834
avg_wait_time                      26.811962
revenue                         12818.300696
cost                            14488.055691
profit                          -1669.754995
profit_per_request                 -0.141649
profit_per_converted_rider         -0.243405
matched_pairs                    2740.000000
Name: historical_baseline, dtype: float64

In [7]:
shadow_results = evaluate_policy(wait_grid=WAIT_GRID, n_replications=N_REPLICATIONS, base_seed=RANDOM_SEED)
shadow_results


,max_wait_seconds,n_replications,avg_quote,expected_conversion_rate,simulated_conversion_rate,shared_rate_given_conversion,avg_wait_time,revenue,cost,profit,profit_per_request,profit_per_converted_rider,matched_pairs
0,15,1,0.714477,0.310841,0.313879,0.249730,12.134054,8807.744216,7982.795423,824.948793,0.069982,0.222959,462.0
1,60,1,0.706379,0.326797,0.329827,0.533951,34.529835,9301.377721,7865.022355,1436.355366,0.121849,0.369433,1038.0
2,120,1,0.704904,0.329062,0.333135,0.656990,54.882608,9384.763006,7693.198499,1691.564507,0.143499,0.430752,1290.0


In [8]:
comparison = shadow_results.copy()
for column in [
    "avg_quote",
    "expected_conversion_rate",
    "simulated_conversion_rate",
    "shared_rate_given_conversion",
    "avg_wait_time",
    "revenue",
    "cost",
    "profit",
    "profit_per_request",
    "profit_per_converted_rider",
    "matched_pairs",
]:
    comparison[f"delta_vs_baseline__{column}"] = comparison[column] - baseline[column]

comparison


,max_wait_seconds,n_replications,avg_quote,expected_conversion_rate,simulated_conversion_rate,shared_rate_given_conversion,avg_wait_time,revenue,cost,profit,...,delta_vs_baseline__expected_conversion_rate,delta_vs_baseline__simulated_conversion_rate,delta_vs_baseline__shared_rate_given_conversion,delta_vs_baseline__avg_wait_time,delta_vs_baseline__revenue,delta_vs_baseline__cost,delta_vs_baseline__profit,delta_vs_baseline__profit_per_request,delta_vs_baseline__profit_per_converted_rider,delta_vs_baseline__matched_pairs
0,15,1,0.714477,0.310841,0.313879,0.249730,12.134054,8807.744216,7982.795423,824.948793,...,-0.271107,-0.268069,-0.549104,-14.677908,-4010.556480,-6505.260268,2494.703788,0.211631,0.466364,-2278.0
1,60,1,0.706379,0.326797,0.329827,0.533951,34.529835,9301.377721,7865.022355,1436.355366,...,-0.255151,-0.252121,-0.264883,7.717874,-3516.922975,-6623.033336,3106.110361,0.263498,0.612837,-1702.0
2,120,1,0.704904,0.329062,0.333135,0.656990,54.882608,9384.763006,7693.198499,1691.564507,...,-0.252886,-0.248812,-0.141844,28.070646,-3433.537690,-6794.857192,3361.319502,0.285148,0.674157,-1450.0


In [10]:
# ============================================================
# Parameter sweep for embedded GRE policies
# Around 864 parameter combinations
# Only keep parameter sets that satisfy:
#   profit(max_wait=15)  > 824
#   profit(max_wait=60)  > 1438
#   profit(max_wait=120) > 1627
# ============================================================

import itertools
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0) Check sandbox function exists
# ------------------------------------------------------------
try:
    evaluate_policy
except NameError:
    raise NameError(
        "evaluate_policy() is not defined. Please run the sandbox cells first."
    )

# ------------------------------------------------------------
# 1) Parameter grid
#    Total combinations = 3*2*2*3*3*4*2 = 864
# ------------------------------------------------------------
PARAM_GRID = {
    # Pricing
    "BASE_PRICE": [0.70, 0.725, 0.75],          # 3
    "PRICE_FLOOR": [0.60, 0.62],                # 2
    "EMPTY_QUEUE_BUMP": [0.00, 0.02],           # 2

    # Matching
    "MAX_CANDIDATES": [8, 10, 12],              # 3  (starts from 8)
    "BASE_MIN_SAVING_RATIO": [0.08, 0.10, 0.12],   # 3
    "BASE_MIN_ABSOLUTE_SAVING": [0.22, 0.28, 0.35, 0.42],  # 4
    "LARGE_QUEUE_THRESHOLD": [6, 8, 10, 12],        # 2  (at least 15)
}

grid_keys = list(PARAM_GRID.keys())
grid_values = [PARAM_GRID[k] for k in grid_keys]
param_combos = [dict(zip(grid_keys, vals)) for vals in itertools.product(*grid_values)]

print(f"Total parameter combinations: {len(param_combos)}")

# ------------------------------------------------------------
# 2) Fixed evaluation waits and qualification thresholds
# ------------------------------------------------------------
WAIT_GRID = [15, 60, 120]
PROFIT_THRESHOLDS = {
    15: 824,
    60: 1438,
    120: 1627,
}

# ------------------------------------------------------------
# 3) Build tunable policy classes
#    Based on GRE policy, with selected parameters tunable
# ------------------------------------------------------------
def build_policy_classes(params):
    class TunablePricingPolicy:
        BASE_PRICE = params["BASE_PRICE"]
        PRICE_FLOOR = params["PRICE_FLOOR"]
        PRICE_CEILING = 0.81
        PAIR_CACHE = {}

        # DO NOT MODIFY signature
        def __init__(self, c=0.70):
            self.c = c

        @staticmethod
        def get_name():
            return TEAM_NAME

        @staticmethod
        def _clip(value, lower=0.0, upper=1.0):
            return max(lower, min(upper, value))

        @staticmethod
        def _safe_ratio(numerator, denominator):
            if denominator <= 0:
                return 0.0
            return float(numerator) / float(denominator)

        @staticmethod
        def _rider_signature(rider):
            return (
                round(float(rider.pickup_lat), 6),
                round(float(rider.pickup_lon), 6),
                round(float(rider.dropoff_lat), 6),
                round(float(rider.dropoff_lon), 6),
                int(rider.pickup_area),
                int(rider.dropoff_area),
                round(float(rider.arrival_time), 3),
            )

        @classmethod
        def _pair_metrics(cls, rider_i, rider_j):
            sig_i = cls._rider_signature(rider_i)
            sig_j = cls._rider_signature(rider_j)
            cache_key = (sig_i, sig_j) if sig_i <= sig_j else (sig_j, sig_i)

            cached = cls.PAIR_CACHE.get(cache_key)
            if cached is not None:
                return cached

            origin_i = (rider_i.pickup_lat, rider_i.pickup_lon)
            dest_i = (rider_i.dropoff_lat, rider_i.dropoff_lon)
            origin_j = (rider_j.pickup_lat, rider_j.pickup_lon)
            dest_j = (rider_j.dropoff_lat, rider_j.dropoff_lon)

            trip_length, shared_length, i_solo_length, j_solo_length, _ = populate_shared_ride_lengths(
                origin_i, dest_i, origin_j, dest_j
            )

            total_length = rider_i.solo_length + rider_j.solo_length
            min_length = min(rider_i.solo_length, rider_j.solo_length)
            solo_ratio_i = cls._safe_ratio(i_solo_length, rider_i.solo_length)
            solo_ratio_j = cls._safe_ratio(j_solo_length, rider_j.solo_length)

            metrics = {
                "trip_ratio": cls._safe_ratio(trip_length, total_length),
                "shared_ratio": cls._safe_ratio(shared_length, min_length),
                "solo_ratio_max": max(solo_ratio_i, solo_ratio_j),
                "absolute_savings": max(0.0, total_length - trip_length),
                "savings_ratio": max(0.0, 1.0 - cls._safe_ratio(trip_length, total_length)),
                "same_pickup": int(rider_i.pickup_area == rider_j.pickup_area),
                "same_dropoff": int(rider_i.dropoff_area == rider_j.dropoff_area),
            }

            if len(cls.PAIR_CACHE) > 25000:
                cls.PAIR_CACHE.clear()

            cls.PAIR_CACHE[cache_key] = metrics
            return metrics

        @classmethod
        def _shareability_snapshot(cls, state, rider):
            same_route = 0
            same_pickup = 0
            same_dropoff = 0
            viable_count = 0
            strong_count = 0
            best_value = 0.0

            for waiting_rider in state:
                if int(waiting_rider.pickup_area) == int(rider.pickup_area):
                    same_pickup += 1
                if int(waiting_rider.dropoff_area) == int(rider.dropoff_area):
                    same_dropoff += 1
                if (
                    int(waiting_rider.pickup_area) == int(rider.pickup_area)
                    and int(waiting_rider.dropoff_area) == int(rider.dropoff_area)
                ):
                    same_route += 1

                metrics = cls._pair_metrics(rider, waiting_rider)

                if metrics["shared_ratio"] <= 0.0:
                    continue
                if metrics["savings_ratio"] < 0.10 and metrics["absolute_savings"] < 0.25:
                    continue
                if metrics["solo_ratio_max"] > 0.80 and metrics["absolute_savings"] < 0.45:
                    continue

                value = (
                    1.80 * metrics["savings_ratio"]
                    + 0.80 * min(metrics["shared_ratio"], 1.0)
                    + 0.45 * max(0.0, 1.0 - metrics["solo_ratio_max"])
                    + 0.12 * metrics["absolute_savings"]
                    + 0.05 * metrics["same_pickup"]
                    + 0.05 * metrics["same_dropoff"]
                )

                viable_count += 1
                if value >= 0.95:
                    strong_count += 1
                best_value = max(best_value, value)

            return {
                "same_route": same_route,
                "same_pickup": same_pickup,
                "same_dropoff": same_dropoff,
                "viable_count": viable_count,
                "strong_count": strong_count,
                "best_value": best_value,
            }

        @classmethod
        def _baseline_price(cls, rider):
            price = cls.BASE_PRICE

            if rider.solo_length < 1.5:
                price -= 0.040
            elif rider.solo_length < 3.0:
                price -= 0.018
            elif rider.solo_length > 8.5:
                price += 0.032
            elif rider.solo_length > 6.0:
                price += 0.018

            return price

        def pricing_function(self, state, rider):
            price = self._baseline_price(rider)
            state_size = len(state)

            if state_size == 0:
                price += params["EMPTY_QUEUE_BUMP"]
                return float(self._clip(price, self.PRICE_FLOOR, self.PRICE_CEILING))

            snapshot = self._shareability_snapshot(state, rider)

            # fixed GRE discounts
            if snapshot["best_value"] >= 1.10:
                price -= 0.028
            elif snapshot["best_value"] >= 0.88:
                price -= 0.018
            elif snapshot["best_value"] >= 0.68:
                price -= 0.010

            if snapshot["same_route"] > 0:
                price -= 0.010
            elif snapshot["same_pickup"] > 0 and snapshot["same_dropoff"] > 0:
                price -= 0.008

            if snapshot["viable_count"] >= 2:
                price -= 0.008
            if snapshot["strong_count"] >= 2:
                price -= 0.006

            if state_size >= 12 and snapshot["viable_count"] == 0:
                price += 0.008
            if state_size >= 30 and snapshot["viable_count"] == 0:
                price += 0.008
            elif state_size >= 30 and snapshot["viable_count"] >= 3:
                price -= 0.006

            return float(self._clip(price, self.PRICE_FLOOR, self.PRICE_CEILING))

    class TunableMatchingPolicy:
        # DO NOT MODIFY signature
        def __init__(self, c=0.70):
            self.c = c

        @staticmethod
        def get_name():
            return TEAM_NAME

        @staticmethod
        def get_origin(r):
            return (r.pickup_lat, r.pickup_lon)

        @staticmethod
        def get_destination(r):
            return (r.dropoff_lat, r.dropoff_lon)

        @staticmethod
        def quick_pair_score(r1, r2):
            pickup_gap = abs(r1.pickup_area - r2.pickup_area)
            dropoff_gap = abs(r1.dropoff_area - r2.dropoff_area)
            time_gap = abs(r1.arrival_time - r2.arrival_time)
            length_gap = abs(r1.solo_length - r2.solo_length)
            return 2.0 * pickup_gap + 2.0 * dropoff_gap + 0.003 * time_gap + 0.2 * length_gap

        def candidate_exact_match_score(self, rider, waiting_rider):
            trip_length, shared_length, i_solo_length, j_solo_length, trip_order = populate_shared_ride_lengths(
                self.get_origin(rider), self.get_destination(rider),
                self.get_origin(waiting_rider), self.get_destination(waiting_rider)
            )

            solo_sum = rider.solo_length + waiting_rider.solo_length
            saving = solo_sum - trip_length
            saving_ratio = saving / max(solo_sum, 1e-6)
            shared_ratio = shared_length / max(trip_length, 1e-6)
            detour_penalty = (i_solo_length + j_solo_length) / max(trip_length, 1e-6)

            score = 2.2 * saving_ratio + 0.35 * shared_ratio + 0.18 * saving - 0.18 * detour_penalty

            return {
                "saving": saving,
                "savings_ratio": saving_ratio,
                "shared_ratio": shared_ratio,
                "detour_penalty": detour_penalty,
                "score": score,
            }

        def matching_function(self, state, rider):
            if len(state) == 0:
                return None

            max_candidates = params["MAX_CANDIDATES"]
            base_min_saving_ratio = params["BASE_MIN_SAVING_RATIO"]
            base_min_absolute_saving = params["BASE_MIN_ABSOLUTE_SAVING"]
            large_queue_threshold = params["LARGE_QUEUE_THRESHOLD"]

            candidates = sorted(state, key=lambda w: self.quick_pair_score(rider, w))[:max_candidates]

            min_saving_ratio = base_min_saving_ratio
            min_absolute_saving = base_min_absolute_saving

            if len(state) >= large_queue_threshold:
                min_saving_ratio -= 0.02
                min_absolute_saving -= 0.08

            if rider.arrival_time > 3300:
                min_saving_ratio -= 0.015
                min_absolute_saving -= 0.05

            min_saving_ratio = max(0.03, min_saving_ratio)
            min_absolute_saving = max(0.08, min_absolute_saving)

            best_candidate = None
            best_score = -1e18

            for w in candidates:
                stats = self.candidate_exact_match_score(rider, w)

                saving = stats["saving"]
                saving_ratio = stats["savings_ratio"]
                score = stats["score"]

                if saving_ratio >= min_saving_ratio and saving >= min_absolute_saving:
                    if score > best_score:
                        best_score = score
                        best_candidate = w

            return best_candidate

    return TunablePricingPolicy, TunableMatchingPolicy

# ------------------------------------------------------------
# 4) Evaluate one parameter combo
# ------------------------------------------------------------
def evaluate_param_combo(params, n_replications=None, base_seed=None):
    pricing_cls, matching_cls = build_policy_classes(params)

    # Inject into sandbox globals
    globals()["StudentPricingPolicy"] = pricing_cls
    globals()["StudentMatchingPolicy"] = matching_cls

    kwargs = {"wait_grid": WAIT_GRID}
    if n_replications is not None:
        kwargs["n_replications"] = n_replications
    if base_seed is not None:
        kwargs["base_seed"] = base_seed

    result_df = evaluate_policy(**kwargs).copy()

    for k, v in params.items():
        result_df[k] = v

    return result_df

# ------------------------------------------------------------
# 5) Qualification check
# ------------------------------------------------------------
def qualifies_by_profit_thresholds(result_df):
    """
    Keep this parameter set only if all three thresholds are met:
      max_wait=15  -> profit > 824
      max_wait=60  -> profit > 1438
      max_wait=120 -> profit > 1627
    """
    profit_by_wait = dict(zip(result_df["max_wait_seconds"], result_df["profit"]))

    for wait, threshold in PROFIT_THRESHOLDS.items():
        if wait not in profit_by_wait:
            return False
        if profit_by_wait[wait] <= threshold:
            return False

    return True

# ------------------------------------------------------------
# 6) Run all combinations, but only record qualified sets
# ------------------------------------------------------------
def run_filtered_param_grid(n_replications=None, base_seed=None):
    qualified_frames = []
    qualified_best_rows = []
    qualified_count = 0

    for i, params in enumerate(param_combos, start=1):
        print(f"[{i}/{len(param_combos)}] Running params: {params}")

        combo_df = evaluate_param_combo(
            params=params,
            n_replications=n_replications,
            base_seed=base_seed,
        )

        if qualifies_by_profit_thresholds(combo_df):
            qualified_count += 1
            qualified_frames.append(combo_df)

            best_row = combo_df.loc[combo_df["profit"].idxmax()].copy()
            qualified_best_rows.append(best_row)

    if len(qualified_frames) == 0:
        print("\nNo parameter set met all thresholds.")
        return pd.DataFrame(), pd.DataFrame()

    all_results_df = pd.concat(qualified_frames, ignore_index=True)
    best_by_combo_df = (
        pd.DataFrame(qualified_best_rows)
        .sort_values("profit", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nQualified parameter sets: {qualified_count}")

    return all_results_df, best_by_combo_df

# ------------------------------------------------------------
# 7) Run
# ------------------------------------------------------------

# Faster test version:
# all_results_df, best_by_combo_df = run_filtered_param_grid(
#     n_replications=5,
#     base_seed=42
# )

# Full version:
all_results_df, best_by_combo_df = run_filtered_param_grid()

# ------------------------------------------------------------
# 8) Save CSV
# ------------------------------------------------------------
all_results_df.to_csv("qualified_grid_search_all_results_local.csv", index=False)
best_by_combo_df.to_csv("qualified_grid_search_best_per_param_combo_local.csv", index=False)

print("\nSaved:")
print("  qualified_grid_search_all_results_local.csv")
print("  qualified_grid_search_best_per_param_combo_local.csv")

display(best_by_combo_df.head(20))

Total parameter combinations: 1728
[1/1728] Running params: {'BASE_PRICE': 0.7, 'PRICE_FLOOR': 0.6, 'EMPTY_QUEUE_BUMP': 0.0, 'MAX_CANDIDATES': 8, 'BASE_MIN_SAVING_RATIO': 0.08, 'BASE_MIN_ABSOLUTE_SAVING': 0.22, 'LARGE_QUEUE_THRESHOLD': 6}
[2/1728] Running params: {'BASE_PRICE': 0.7, 'PRICE_FLOOR': 0.6, 'EMPTY_QUEUE_BUMP': 0.0, 'MAX_CANDIDATES': 8, 'BASE_MIN_SAVING_RATIO': 0.08, 'BASE_MIN_ABSOLUTE_SAVING': 0.22, 'LARGE_QUEUE_THRESHOLD': 8}
[3/1728] Running params: {'BASE_PRICE': 0.7, 'PRICE_FLOOR': 0.6, 'EMPTY_QUEUE_BUMP': 0.0, 'MAX_CANDIDATES': 8, 'BASE_MIN_SAVING_RATIO': 0.08, 'BASE_MIN_ABSOLUTE_SAVING': 0.22, 'LARGE_QUEUE_THRESHOLD': 10}
[4/1728] Running params: {'BASE_PRICE': 0.7, 'PRICE_FLOOR': 0.6, 'EMPTY_QUEUE_BUMP': 0.0, 'MAX_CANDIDATES': 8, 'BASE_MIN_SAVING_RATIO': 0.08, 'BASE_MIN_ABSOLUTE_SAVING': 0.22, 'LARGE_QUEUE_THRESHOLD': 12}
[5/1728] Running params: {'BASE_PRICE': 0.7, 'PRICE_FLOOR': 0.6, 'EMPTY_QUEUE_BUMP': 0.0, 'MAX_CANDIDATES': 8, 'BASE_MIN_SAVING_RATIO': 0.08, 'BASE

,max_wait_seconds,n_replications,avg_quote,expected_conversion_rate,simulated_conversion_rate,shared_rate_given_conversion,avg_wait_time,revenue,cost,profit,profit_per_request,profit_per_converted_rider,matched_pairs,BASE_PRICE,PRICE_FLOOR,EMPTY_QUEUE_BUMP,MAX_CANDIDATES,BASE_MIN_SAVING_RATIO,BASE_MIN_ABSOLUTE_SAVING,LARGE_QUEUE_THRESHOLD
0,120.0,1.0,0.705248,0.328283,0.330506,0.671971,52.883727,9294.765041,7656.359752,1638.405289,0.138989,0.420535,1309.0,0.725,0.60,0.02,12.0,0.12,0.35,8.0
1,120.0,1.0,0.705248,0.328283,0.330506,0.671971,52.883727,9294.765041,7656.359752,1638.405289,0.138989,0.420535,1309.0,0.725,0.62,0.02,12.0,0.12,0.35,8.0
2,120.0,1.0,0.705309,0.328155,0.329997,0.674550,52.648329,9281.724695,7647.569544,1634.155151,0.138629,0.420091,1312.0,0.725,0.60,0.02,12.0,0.12,0.22,8.0
3,120.0,1.0,0.705309,0.328155,0.329997,0.674550,52.648329,9281.724695,7647.569544,1634.155151,0.138629,0.420091,1312.0,0.725,0.60,0.02,12.0,0.12,0.28,8.0
4,120.0,1.0,0.705309,0.328155,0.329997,0.674550,52.648329,9281.724695,7647.569544,1634.155151,0.138629,0.420091,1312.0,0.725,0.62,0.02,12.0,0.12,0.22,8.0
5,120.0,1.0,0.705309,0.328155,0.329997,0.674550,52.648329,9281.724695,7647.569544,1634.155151,0.138629,0.420091,1312.0,0.725,0.62,0.02,12.0,0.12,0.28,8.0
6,120.0,1.0,0.705143,0.328374,0.330675,0.672140,53.015906,9291.200463,7663.230049,1627.970415,0.138104,0.417642,1310.0,0.725,0.60,0.02,8.0,0.12,0.35,8.0
7,120.0,1.0,0.705143,0.328374,0.330675,0.672140,53.015906,9291.200463,7663.230049,1627.970415,0.138104,0.417642,1310.0,0.725,0.62,0.02,8.0,0.12,0.35,8.0
